# Project Summary: GaleMed Insights – A RAG-Based Medical Chatbot Using Pinecone and LLMs

---
## Project Overview
GaleMed Insights is an advanced medical chatbot built using a Retrieval-Augmented Generation (RAG) framework, leveraging the wealth of medical knowledge contained in The Gale Encyclopedia of Medicine, which spans over 637 pages. By combining PDF processing, semantic embedding, and vector databases, GaleMed Insights offers accurate, detailed, and user-friendly responses to medical inquiries. This makes it a valuable and accessible resource for individuals seeking trustworthy health information.

With its foundation in The Gale Encyclopedia of Medicine, GaleMed Insights ensures users can access well-researched, vetted content on a wide range of medical topics, from conditions and treatments to best healthcare practices. The chatbot empowers users to easily navigate complex medical concepts and obtain clear, relevant answers.

---

## Why Use RAG-Based Methodology Instead of Just LLMs?
Using a Retrieval-Augmented Generation (RAG) methodology offers significant advantages over traditional large language models (LLMs) alone:

#### Higher Accuracy and Reliability:
GaleMed Insights retrieves factual information directly from The Gale Encyclopedia of Medicine, ensuring that responses are factually accurate and relevant. This drastically reduces the risk of generating incorrect information (known as "hallucinations"), which is a common issue when using LLMs on their own.

#### Targeted Knowledge Base:
Instead of relying on broad datasets, GaleMed Insights focuses on a specific medical knowledge base, ensuring that users receive expert, specialized responses. This contrasts with general LLMs, which may provide superficial or irrelevant answers to health-related queries.

#### Contextual Relevance:
Through chunking and semantic embedding, GaleMed Insights retrieves the most relevant information while maintaining the necessary context, which leads to clearer and more coherent responses tailored to the user's needs.

#### Efficient Memory Management:
Dividing the text into chunks avoids running into token limits of LLMs, allowing the model to process large documents effectively without losing important details from the encyclopedia.

#### Improved User Experience:
With precise, well-formatted responses, GaleMed Insights enhances the readability and accessibility of complex medical information, making it easier for users to understand and act on the information provided.

---
## Why This Project?
The GaleMed Insights project is designed to address the growing need for reliable, accurate, and accessible medical information. In a world where misinformation spreads easily, especially on health-related topics, GaleMed Insights offers a trusted source of knowledge from The Gale Encyclopedia of Medicine. Key reasons for developing this project include:

#### Improving Health Literacy:
GaleMed Insights empowers individuals to make informed decisions about their health by providing clear and factual medical information.

#### Supporting Healthcare Professionals:
The chatbot serves as a quick reference tool for healthcare providers, giving them rapid access to accurate medical information to assist in patient care.

#### Easy Knowledge Base Updates:
As medical knowledge evolves, GaleMed Insights can easily be updated with new data, ensuring the chatbot remains current and provides the latest medical insights.

---
## Steps: 
### Document Reading with PyPDF:
The project utilizes the pyPDF library to read and extract data from The Gale Encyclopedia of Medicine. This library efficiently handles large documents, facilitating the structured extraction of valuable medical information.

### Data Chunking:
After extracting the data, it is crucial to divide the information into manageable chunks. Chunking helps to:

- Improve retrieval efficiency by enabling quick access to relevant sections.

- Enhance context provided in responses, allowing the model to generate more meaningful answers.

- Mitigate issues related to the maximum token limits of language models by ensuring that only concise and relevant information is processed at any given time.

### Creating Semantic Embeddings:

- To facilitate meaningful queries and responses, we employ the HuggingFaceEmbeddings model (sentence-transformers/all-MiniLM-L6-v2). This model generates semantic embeddings, which capture the context and meaning of the text beyond surface-level words. These embeddings enable GaleMed to comprehend user queries more effectively and retrieve the most relevant information.

### Building a Vector Database with Pinecone:
The next step involves creating a vector database using Pinecone. We load the chunked and embedded data into Pinecone, establishing a knowledge base that can efficiently manage user queries by identifying similar vectors and retrieving pertinent information.

### Testing Queries:
Once the knowledge base is established, we conduct test queries to evaluate the performance of GaleMed. This testing phase ensures that the chatbot retrieves accurate and relevant information effectively, confirming the effectiveness of the RAG approach.

### Utilizing LLM for Response Formatting:
To enhance the clarity and readability of the information returned to users, we utilize a large language model (LLM) based on the LLaMA architecture (CTransformers). This offline version processes the extracted data and formats it into coherent responses, making complex medical information accessible and understandable.

### Flask Application Development:
Finally, we aim to develop a Flask web application that serves as the interface for GaleMed. This application will allow users to interact with the chatbot seamlessly, posing questions and receiving comprehensive answers based on the medical encyclopedia.

---
## Real-World Use Cases of RAG Methodology in Other Fields
This RAG-based approach is not only useful in healthcare but also has real-world applications in various fields, such as:

#### Legal Industry:
Legal research chatbots can use RAG to pull up relevant laws, precedents, and case summaries from specific legal databases, providing accurate and contextual legal advice.

#### Customer Support:
Companies can use RAG to power customer service bots that retrieve relevant product information and troubleshooting steps from their internal knowledge bases, enhancing customer experience with quicker and more accurate responses.

#### Education:
RAG-based educational bots can assist students by providing targeted explanations and detailed answers based on textbooks, enhancing their learning experiences.

#### Technical Documentation:
Tech companies can use RAG to help engineers and developers quickly retrieve information from large technical manuals or knowledge bases to solve problems efficiently.

---

By using a RAG framework such as GaleMed Insights, we can enable and ensure that users get precise, relevant information from a trusted source, improving the overall accuracy and reliability of responses across various industries.

Data got from https://www.academia.edu/32752835/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND_EDITION

In [1]:
print("OK!")

OK!


In [2]:
import os
import json
import time
from pinecone import Pinecone, ServerlessSpec
from langchain import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore as LangchainPinecone  # Updated import
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.llms import CTransformers


/Users/parthawgoswami/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/parthawgoswami/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Initializing index name and the Pinecone

os.environ["PINECONE_API_KEY"] = "pcsk_4ArCC1_Rzav3ZitfjUHYBhYFxYqkKBfVtqDyWUvH3ozBWD46nFViEpjBdx9djfBJJh8FLc"

index_name="rag-knowledge"

# Initialize Pinecone with optional parameters
try:
    pc = Pinecone(
        api_key=os.environ.get("PINECONE_API_KEY"),
        proxy_url=None,            # Example optional parameter
        proxy_headers=None,        # Example optional parameter
        ssl_ca_certs=None,        # Example optional parameter
        ssl_verify=True,  # Example optional parameter, usually set to True
    )
    
    # Check if the index exists
    indexes = pc.list_indexes()  # List of index names
    index_names = indexes.names()  # Get only the names of the indexes

    if index_name not in index_names:
        print(f'{index_name} does not exist')
        # Change the following line to create the index of your choice
        pc.create_index(
             name=index_name,
             dimension=384,
             metric="cosine",
             spec=ServerlessSpec(
                 cloud="aws",
                 region="us-east-1"
             )
         )
    else:
        print(f'{index_name} exists.')

    # Connect to the existing index
    index = pc.Index(index_name)

except Exception as e:
    print(f"An error occurred while checking indexes: {e}")

# Embedding the text chunks and storing them in Pinecone
try:
    docsearch = LangchainPinecone.from_texts(
        texts=[t.page_content for t in text_chunks],  # Assuming `text_chunks` is a list of text splits
        embedding=embeddings,  # Embedding model instance
        index_name=index_name
    )
except Exception as e:
    print(f"An error occurred while creating embeddings: {e}")

rag-knowledge exists.
An error occurred while creating embeddings: name 'text_chunks' is not defined


In [4]:
# from pinecone import Pinecone

pc = Pinecone(api_key="pcsk_4ArCC1_Rzav3ZitfjUHYBhYFxYqkKBfVtqDyWUvH3ozBWD46nFViEpjBdx9djfBJJh8FLc")
pc.list_indexes()

[
    {
        "name": "rag-knowledge",
        "metric": "cosine",
        "host": "rag-knowledge-oaavv06.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 768,
        "deletion_protection": "disabled",
        "tags": null
    }
]

### Vector stores are there in the Pinecone that we created

In [10]:
from langchain.schema import Document

# Load vectors from the exported Pinecone JSON and convert to LangChain Documents
def load_json(data):
    with open(data, "r") as f:
        raw = json.load(f)
    
    vectors = raw.get("vectors", raw)  # support both {"vectors": [...]} and plain list
    if isinstance(vectors, dict):
        vectors = list(vectors.values())
    
    documents = []
    for v in vectors:
        text = v.get("metadata", {}).get("text", "")
        metadata = {k: val for k, val in v.get("metadata", {}).items() if k != "text"}
        if text:
            documents.append(Document(page_content=text, metadata=metadata))
    
    print(f"Loaded {len(documents)} documents from JSON")
    return documents


In [11]:
extracted_data = load_json("/Users/parthawgoswami/Documents/ADV_NLP/pinecone_all_vectors.json")

Loaded 45752 documents from JSON


In [12]:
#extracted_data

In [13]:
# The JSON data is already chunked (each vector's metadata.text is a chunk).
# We return the documents directly without re-splitting.
def text_split(extracted_data):
    return extracted_data


In [14]:
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 45752


In [15]:
# text_chunks

In [26]:
# Using all-mpnet-base-v2 which produces 768-dim embeddings to match the index
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
    return embeddings


In [27]:
embeddings = download_hugging_face_embeddings()

In [18]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [28]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 768


In [20]:
# query_result

### Writing the Data into Pinecone

In [21]:
#Initializing index name and the Pinecone

os.environ["PINECONE_API_KEY"] = "pcsk_4ArCC1_Rzav3ZitfjUHYBhYFxYqkKBfVtqDyWUvH3ozBWD46nFViEpjBdx9djfBJJh8FLc"

index_name="rag-knowledge"

# Initialize Pinecone with optional parameters
try:
    pc = Pinecone(
        api_key=os.environ.get("PINECONE_API_KEY"),
        proxy_url=None,            # Example optional parameter
        proxy_headers=None,        # Example optional parameter
        ssl_ca_certs=None,        # Example optional parameter
        ssl_verify=True,  # Example optional parameter, usually set to True
    )
    
    time.sleep(2)  # Optional sleep to ensure initialization completes

    # Check if the index exists
    indexes = pc.list_indexes()  # List of index names
    index_names = indexes.names()  # Get only the names of the indexes

    if index_name not in index_names:
        print(f'{index_name} does not exist')
        
    else:
        print(f'{index_name} exists.')

    # Connect to the existing index
    index = pc.Index(index_name)

except Exception as e:
    print(f"An error occurred while checking indexes: {e}")


rag-knowledge exists.


In [22]:
import json

# ── Load raw vectors (already embedded) from JSON ────────────────────────────
print("Loading vectors from JSON...")
with open("/Users/parthawgoswami/Documents/ADV_NLP/pinecone_all_vectors.json") as f:
    raw = json.load(f)

vectors = raw.get("vectors", raw)
if isinstance(vectors, dict):
    vectors = list(vectors.values())

actual_dim = len(vectors[0]["values"])
print(f"Total vectors: {len(vectors)}, dimension: {actual_dim}")

# ── Delete and recreate index with the correct dimension ──────────────────────
print("\nDeleting existing index...")
pc.delete_index(index_name)
time.sleep(8)

print(f"Recreating index with dimension={actual_dim}...")
pc.create_index(
    name=index_name,
    dimension=actual_dim,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)
time.sleep(8)
index = pc.Index(index_name)
print("Index ready.")

# ── Slim metadata to stay under Pinecone's 40KB-per-vector limit ─────────────
KEEP_KEYS = {"text", "source_files", "node_type", "level"}
MAX_TEXT_CHARS = 4000

def slim_metadata(meta):
    slimmed = {}
    for k in KEEP_KEYS:
        if k in meta:
            v = meta[k]
            if k == "text" and isinstance(v, str):
                v = v[:MAX_TEXT_CHARS]
            elif k == "source_files" and isinstance(v, list):
                v = v[:5]
            slimmed[k] = v
    return slimmed

# ── Upsert in batches of 100 ──────────────────────────────────────────────────
BATCH_SIZE = 100
skipped = 0
for i in range(0, len(vectors), BATCH_SIZE):
    batch = vectors[i : i + BATCH_SIZE]
    upsert_data = []
    for v in batch:
        if not v.get("values"):
            skipped += 1
            continue
        upsert_data.append((v["id"], v["values"], slim_metadata(v.get("metadata", {}))))
    index.upsert(vectors=upsert_data)
    if i % 5000 == 0:
        print(f"  Upserted {min(i + BATCH_SIZE, len(vectors))} / {len(vectors)}")

time.sleep(5)
stats = index.describe_index_stats()
print(f"\nDone! Total vectors in index: {stats.get('total_vector_count', 0)}")
if skipped:
    print(f"Skipped {skipped} vectors with no values.")

# NOTE: queries must now use a 768-dim embedding model to match the index
docsearch = LangchainPinecone.from_existing_index(index_name, embeddings, text_key="text")
print("docsearch ready.")


Loading vectors from JSON...
Total vectors: 45752, dimension: 768

Deleting existing index...
Total vectors: 45752, dimension: 768

Deleting existing index...
Recreating index with dimension=768...
Recreating index with dimension=768...
Index ready.
Index ready.
  Upserted 100 / 45752
  Upserted 100 / 45752
  Upserted 5100 / 45752
  Upserted 5100 / 45752
  Upserted 10100 / 45752
  Upserted 10100 / 45752
  Upserted 15100 / 45752
  Upserted 15100 / 45752
  Upserted 20100 / 45752
  Upserted 20100 / 45752
  Upserted 25100 / 45752
  Upserted 25100 / 45752
  Upserted 30100 / 45752
  Upserted 30100 / 45752
  Upserted 35100 / 45752
  Upserted 35100 / 45752
  Upserted 40100 / 45752
  Upserted 40100 / 45752
  Upserted 45100 / 45752
  Upserted 45100 / 45752

Done! Total vectors in index: 45752

Done! Total vectors in index: 45752
docsearch ready.
docsearch ready.


### Testing it out

In [29]:
index_name = "rag-knowledge"
# Load existing index — text_key="text" matches the metadata field used during upsert
docsearch = LangchainPinecone.from_existing_index(index_name, embeddings, text_key="text")

query = "What are Allergies"
docs = docsearch.similarity_search(query, k=3)
print("Result", docs)


Result [Document(id='chunk_mateescu2001_d9e9f3d7_0118', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/mateescu2001.pdf']}, page_content='Assume that the Parikh matrix of w is Ψ Mk (w)=( mi,j)1≤ i,j≤ k+1 , and that [Ψ Mk (w)]− 1 =( m′ i,j)1≤ i,j≤ k+1 .T h e n |mi(w)|scatt− ai,j = |(m′ i,j+1)| for all 1 ≤ i, j ≤ k. 4. Computing the inverse of a Parikh matrix We consider now another method to compute the inverse of a Parikh matrix.'), Document(id='chunk_li2020_9d5edbd2_0056', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/li2020.pdf']}, page_content='On the surface, this task appears to be straightforward. One can translate QAs from Jennifer into another language using machine translation (ML). However, language ex- pansion needs to overcome several major obstacles: • Language Fluency: Results produced by com- mercially available ML services still require signiﬁcant manual reﬁnement, particularly 

In [30]:
query = "Cure for acne?"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='chunk_zeng2026_6e55821f_0017', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/student_papers/zeng2026.pdf']}, page_content='Yet, despite their growing ca- pability to participate in simulated environments, leveraging LLM agents to optimize the rules of those environments remains almost entirely underexplored. We introduceRuleSmith, a general framework for auto- matic balancing of asymmetric games through LLM-driven self-play and Bayesian optimization. As a proof of con- cept, we crafed an asymmetric strategy game,CivMini, a fully parameterized grid turn-based game inspired by classic civilization-style mechanics.'), Document(id='chunk_chen2024_bb4db8ea_0034', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/chen2024.pdf']}, page_content='• Counterfactual Robustness, which evaluates whether LLMs can identify risks of known factual errors in the retrieved documents when the LLMs are given 

In [31]:
query = "I have pain in my head"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='chunk_shi2023_d9881044_0267', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/shi2023.pdf']}, page_content='Prompt for MMLU Knowledge: received 122,000 buys (excluding WWE Network views), down from the previous year´s 199,000 buys. The event is named after the Money In The Bank ladder match, in which multiple wrestlers use ladders to retrieve a briefcase hanging above the ring. The winner is guaranteed a match for the WWE World Heavyweight Championship at a time of their choosing within the next year.'), Document(id='chunk_shao2011_94d55807_0035', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/shao2011.pdf']}, page_content='H is an efﬁcient indexing function to map each xi from d-dimensional Euclidean space to m-dimensional Hamming space yi, we deﬁne H as follows: H : xi 2 Rd ! yi 2f /C0 1; 1gm ð1Þ The indexing function H deﬁned above have following good charac- teristics: (1

In [32]:
docs

[Document(id='chunk_shi2023_d9881044_0267', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/shi2023.pdf']}, page_content='Prompt for MMLU Knowledge: received 122,000 buys (excluding WWE Network views), down from the previous year´s 199,000 buys. The event is named after the Money In The Bank ladder match, in which multiple wrestlers use ladders to retrieve a briefcase hanging above the ring. The winner is guaranteed a match for the WWE World Heavyweight Championship at a time of their choosing within the next year.'),
 Document(id='chunk_shao2011_94d55807_0035', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/shao2011.pdf']}, page_content='H is an efﬁcient indexing function to map each xi from d-dimensional Euclidean space to m-dimensional Hamming space yi, we deﬁne H as follows: H : xi 2 Rd ! yi 2f /C0 1; 1gm ð1Þ The indexing function H deﬁned above have following good charac- teristics: (1) H is

In [33]:
query = "I am sad all the time"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='chunk_sarikaya2014_382e8b31_0101', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/sarikaya2014.pdf']}, page_content='The MaxEnt models are trained using the improved iterative scaling algorithm [21] with Gaussian prior smoothing [20] using a single universal variance parameter of 2.0. B. Boosting Boosting is a method that can be used in conjunction with many learning algorithms to improve the accuracy of the learning algorithm. The idea of Boosting is to produce an accu- rate prediction rule by combining many moderately inaccurate (weak) rules into a single classiﬁer.'), Document(id='chunk_collobert2011_90f74f5a_0153', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/papers/collobert2011.pdf']}, page_content='If no non-linearity is introduced, our network would be a simple linear model. We chose a “hard” version of the hyperbolic tangent asnon-linearity. It has the advantage of being sli

In [34]:
query = "What to do if you are sad?"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='chunk_collobert2011_90f74f5a_0153', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/papers/collobert2011.pdf']}, page_content='If no non-linearity is introduced, our network would be a simple linear model. We chose a “hard” version of the hyperbolic tangent asnon-linearity. It has the advantage of being slightly cheaper to compute (compared to the exact hype rbolic tangent), while leaving the generalization performance unchanged (Collobert, 2004).'), Document(id='chunk_brinkman2025_42bee949_0173', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/student_papers/brinkman2025.pdf']}, page_content='It is probable that LMs deploy distinct concept spaces from humans; thus, human explanations of neurons or features are likely to be biased in ways that can result in suboptimal predic- tions of when the neuron or feature will activate (cf. Huang et al., 2023). Non-linear features. Sparse autoencoders are a m

In [35]:
query = "Who is superman?"

docs=docsearch.similarity_search(query, k=3)

print("Result", 
      docs)

Result [Document(id='chunk_marin2019_042889f4_0406', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/marin2019.pdf']}, page_content='From 2000 to 2005, he spent postdoctoral training at the Brain and Cognitive Science Department and the Computer Science and Artiﬁcial Intelligence Laboratory, MIT. He is now a Professor of Electrical Engineering and Computer Science at the Massachusetts Institute of Technology (MIT). Prof. Torralba is an Associate Editor of the International Journal in Computer Vision, and has served as program chair for the Computer Vision and Pattern Recognition conference in 2015.'), Document(id='chunk_karpukhin2020_ba91a7cd_0219', metadata={'level': 0.0, 'node_type': 'chunk', 'source_files': ['documents/adv_nlp/aux_papers/karpukhin2020.pdf']}, page_content='Toutanova, Llion Jones, Ming-Wei Chang, Andrew Dai, Jakob Uszkoreit, Quoc Le, and Slav Petrov. 2019. Natu- ral questions: a benchmark for question answering research. T